Note that the following paths (relative to the directory in which the code is run) must exist for the code to run correctly.  Data is assumed to be in these directories:
- "../Data/train"
    - This directory must contain the training data with files within their respective genre folder
- "../Data/test"
    - This directory must contain the test data
- "../Output/"
    - Output files will be written to this directory
- "../Data_Proc/train"
    - Training data will be processed into spectrogram images and saved here
- "../Data_Proc/test"
    - Test data will be processed into spectrogram images and saved here
- "../Eval_Data/train"
    - Training spectrograms will be split into train and test sets. The train split will be put here
- "../Eval_Data/test"
    - Training spectrograms will be split into train and test sets. The test split will be put here

First, load the necessary packages:

In [ ]:
# Import Necessary Packages
import os
import librosa
import numpy as np
import pandas as pd
import seaborn as sn
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import pygad
from IPython.display import Audio, display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from Data_Load_and_Process import (proc_all_au_data, proc_all_au_test_data, 
        get_all_au_test_data, get_all_au_data, get_all_au_data_labels, 
        split_spec_data_for_CNN_test_eval, create_class_dirs_in_eval_dirs,
        train_test_split_CNN_data, move_eval_files, proc_all_au_data_rgb, proc_all_au_test_data_rgb)

Load necessary pytorch packages:

In [ ]:
import torch
#import torch.nn as nn
#import torch.optim as optim
from torch import nn, optim
from torchinfo import summary
from torchmetrics import Accuracy
from torchvision import transforms, datasets
import pytorch_lightning as pl
from torch.utils.data import Dataset
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import DataLoader, TensorDataset, random_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning import LightningModule, LightningDataModule, Trainer, seed_everything

Initialize necessary variables (including paths):

In [ ]:
# Features to extract from raw training data:
selected_features = ["mfcc", "chroma", "spectral contrast", "zero crossings"]
selected_features = ["stft"]
selected_features = ["mfcc"]
selected_features = ["melspec"]

# Data input and output directories:
raw_data_path = "../Data"
train_data_path = "../Data/train"
test_data_path = "../Data/test"
sample_input_path = "../Data/sandbox"
debug_data_directory = sample_input_path
proc_data_path = "../Data_Proc"
proc_train_data_path = "../Data_Proc/train"
proc_test_data_path = "../Data_Proc/test"
eval_data_path = "../Eval_Data"
eval_train_data_path = "../Eval_Data/train"
eval_test_data_path = "../Eval_Data/test"

Load data from audio files, extract features, stack together and save as images in a processed data directory ("../Data_Proc").  Then split training data into train and test sets for evaluation ("../Eval_Data").  This data will be used in CNN evaluation and transfer learning evaluation below.

In [ ]:
# Load, feature extract, save as image:
proc_all_au_data(selected_features)
proc_all_au_test_data(selected_features)

# Get data labels
Y, encoder = get_all_au_data_labels()


# Get list of paths and processed data (for CNN)
class_list, path_list = split_spec_data_for_CNN_test_eval()

# Create class directories in evaluation directories:
create_class_dirs_in_eval_dirs(class_list)

# Split evaluation data lists into training and test lists
trainpath_list, testpath_list, trainclass_list, testclass_list = train_test_split_CNN_data(class_list, path_list)

# Move processed train files to evaluation train and test directories
move_eval_files(trainpath_list, trainclass_list, eval_train_data_path)
move_eval_files(testpath_list, testclass_list, eval_test_data_path)

### CNN Section:

Define necessary classes:
1. First, since datasets.ImageFolder() requires a directory structure where data is organized into classes, we cannot use it to load the test data.  We therefore define ClasslessImageFolder() to use for loading the test data
2. We define a class for loading and transforming the "audio" spectrogram image data: AudioDataModule() which inherits from LightningDataModule

In [ ]:

class ClasslessImageFolder(Dataset):
    def __init__(self, root, transform=None):
        self.file_paths = sorted(list(Path(root).glob("*.png")))
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        image_data = Image.open(file_path).convert("RGB")
        if self.transform:
            image_data = self.transform(image_data)

        return image_data, file_path.name


class AudioDataModule(LightningDataModule):
    def __init__(self, data_dir=eval_data_path, predict_data_dir=proc_test_data_path, batch_size=64, num_workers=32):
        super().__init__()
        self.data_dir = data_dir
        self.predict_data_dir = predict_data_dir
        self.train_data_dir = os.path.join(data_dir, "train")
        self.test_data_dir = os.path.join(data_dir, "test")
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.transform_train = transforms.Compose([
            transforms.ToTensor()
        ])

        self.transforms_test = transforms.Compose([
            transforms.ToTensor()
        ])

    def prepare_data(self) -> None:
        pass

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            full_dataset = datasets.ImageFolder(root=self.train_data_dir, 
                                                transform=self.transform_train)
            train_size = int(0.9 * len(full_dataset))
            val_size = len(full_dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(full_dataset, 
                                                                [train_size, val_size])
            self.classnames = full_dataset.classes
            self.class_to_idx = full_dataset.class_to_idx
        elif stage == 'test':
            self.test_dataset = datasets.ImageFolder(root=self.test_data_dir, 
                                                     transform=self.transforms_test)
        elif stage == 'predict':
            self.predict_dataset = ClasslessImageFolder(root=self.predict_data_dir, 
                                                     transform=self.transforms_test)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                          shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)
    
    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)
    
    def predict_dataloader(self):
        return DataLoader(self.predict_dataset, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)





## Genetic Algorithm: Adapted from "Genetic CNN" by Xie et al. (2017)

### Portion of the CNN that is dynamically constructed

In [ ]:
class GeneticStage(nn.Module):
    def __init__(self, bit_string, K_s, in_channels, out_channels):
        super().__init__()
        self.K_s = K_s
        self.out_channels = out_channels
        
        # Each node is Conv -> BN -> ReLU
        self.nodes = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                #nn.BatchNorm2d(out_channels),
                nn.ReLU()
            ) for _ in range(K_s)
        ])
        
        # Default entry/exit nodes handle data flow between stages
        self.default_input = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            #nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
        self.default_output = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            #nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
        
        self.connections = self._decode_bits(bit_string)

    def _decode_bits(self, bits):
        # Maps bits to node pairs (i, j) where i < j
        adj = {j: [] for j in range(1, self.K_s + 1)}
        bit_idx = 0
        for j in range(2, self.K_s + 1):
            for i in range(1, j):
                if bits[bit_idx] == 1:
                    adj[j].append(i)
                bit_idx += 1
        return adj

    def forward(self, x):
        outputs = {}
        # Entry: Data from previous stage 
        input_data = self.default_input(x)
        
        # Internal Processing: Nodes sum their inputs before convolution
        for j in range(1, self.K_s + 1):
            predecessors = [outputs[i] for i in self.connections[j]]
            
            # Use default input if node has no predecessors
            node_input = sum(predecessors) if predecessors else input_data
            outputs[j] = self.nodes[j-1](node_input)
            
        # Sum nodes with no successors
        successors = set().union(*self.connections.values())
        final_nodes = [outputs[i] for i in range(1, self.K_s + 1) if i not in successors]
        
        # Special case: all-zero string
        if not final_nodes: return input_data 
            
        return self.default_output(sum(final_nodes))

### CNN construction that accepts GeneticStage

In [ ]:
class EvolvingCNN(LightningModule):
    def __init__(self, bit_string, config):
        super().__init__()
        self.save_hyperparameters()
        S = config['S']
        K_list = config['K_list']
        filters = config['filters']
        pool_kernel = config.get('maxpool_kernel')
        self.acc = Accuracy(task="multiclass", num_classes=10)
        
        # Partition the binary string into stage segments
        segments = []
        start = 0
        for K in K_list:
            length = (K * (K - 1)) // 2
            segments.append(bit_string[start : start + length])
            start += length
            
        # Build the dynamic pipeline
        self.stages = nn.ModuleList()
        current_channels = 3 # RGB input
        for i in range(S):
            self.stages.append(GeneticStage(segments[i], K_list[i], current_channels, filters[i]))
            #self.stages.append(nn.MaxPool2d(kernel_size=2)) # Pooling between stages
            self.stages.append(nn.MaxPool2d(kernel_size=pool_kernel))
            current_channels = filters[i]
            
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, config['num_classes'])
        )

    def forward(self, x):
        #out = self.model[:-5](x)
        #print("Shape before flatten:", out.shape)
        #out = self.model[-5:](out)
        #return out
        #print(x.shape)
        return self.classifier(x)
    
    def shared_step(self, batch, stage=None):
        #print("\nShared Step:", len(batch))
        x, y = batch
        #Uncomment the next two lines if training on gpus.
        x = x.to(self.device)
        y = y.to(self.device)
        #print("Batch x and y:", x.shape, y.shape)
        logits = self(x)
        loss = nn.CrossEntropyLoss()(logits, y)
        preds = logits.argmax(1)
        acc = self.acc(preds, y)
        if stage:
            self.log(f"{stage}_loss", loss, on_epoch=True, prog_bar=True, logger=True)
            self.log(f"{stage}_acc", acc, on_epoch=True, prog_bar=True, logger=True)
        return loss, acc
    
    def training_step(self, batch, batch_idx):
        loss, acc = self.shared_step(batch, "train")
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss, acc = self.shared_step(batch, "val")
        return loss
    
    def test_step(self, batch, batch_idx):
        loss, acc = self.shared_step(batch, "test")
        return loss
    
    def predict_step(self, batch, batch_idx):
        #print("\nPredict Step:")
        x, filenames = batch
        logits = self(x)
        #print("model output type:", type(logits))
        #print("sizes:", logits.shape)
        #print()
        return logits, filenames
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=2, factor=0.5
        )
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler,
                                 "monitor": "val_loss"}}

### Testing

In [ ]:
# Two stage config
config_two_stage = {
    'S': 2, 
    'K_list': [3, 4], 
    'filters': [32, 64], 
    'num_classes': 10,
    'maxpool_kernel': (1,5)
}

# One stage config
config_one_stage = {
    'S': 1, 
    'K_list': [3], 
    'filters': [32], 
    'num_classes': 10,
    'maxpool_kernel': (1,5)
}


#model = EvolvingCNN(bit_string='111111111', config=config_two_stage)
model = EvolvingCNN(bit_string = '111', config=config_one_stage)


trainer = Trainer(
    max_epochs=20, 
    #limit_train_batches=10, # Only train on 10 batches for speed
    accelerator="auto", 
    logger=False, 
    enable_checkpointing=False
)

dm = AudioDataModule(batch_size=64, num_workers=32)
trainer.fit(model, dm)
#metrics = trainer.validate(model, dataloaders=test_loader, verbose=False)
metrics = trainer.validate(model, dm)

### Define functions for PyGad GA

In [ ]:
dm = AudioDataModule(batch_size=64, num_workers=32)

def fitness_func(ga_instance, solution, solution_idx):
    # Configuration for the model
    config = {
        'S': 2, 
        'K_list': [3, 4], 
        'filters': [32, 64], 
        'num_classes': 10,
        'maxpool_kernel': (1,5)
    }
    
    # Instantiate the model with the current "DNA" (solution)
    model = EvolvingCNN(bit_string=solution, config=config)
    
    # Distribute across processes
    process_id = os.getpid()
    available_gpus = [0]
    #gpu_id = available_gpus[process_id % len(available_gpus)]
    #gpu_id = available_gpus[(process_id % 10)] #Try two parallel processes on single gpu
    gpu_id = available_gpus[0]
    
    # Setup trainer
    trainer = Trainer(
        max_epochs=20, 
        #limit_train_batches=10, # Only train on 10 batches for speed
        accelerator="gpu",
        devices=[gpu_id],
        logger=False, 
        enable_checkpointing=False
    )
    
    # Train and validate
    
    #dm = AudioDataModule(batch_size=64, num_workers=32)
    trainer.fit(model, dm)
    metrics = trainer.validate(model, dm)
    
    # Extract accuracy as fitness
    accuracy = metrics[0]['val_acc'] 
    return accuracy

def roulette_wheel_selection(fitness, num_parents, ga_instance):
    # Roulette selection requires non-negative values to calculate probabilities
    if np.min(fitness) < 0:
        fitness = fitness - np.min(fitness) + 1e-6
    
    # Calculate selection probabilities
    fitness_sum = np.sum(fitness)
    if fitness_sum == 0:
        # If all fitnesses are 0, use uniform probability
        probs = np.ones(len(fitness)) / len(fitness)
    else:
        probs = fitness / fitness_sum
    
    # Select indices based on the calculated probabilities
    population_indices = np.arange(len(fitness))
    selected_indices = np.random.choice(
        population_indices, 
        size=num_parents, 
        replace=True, # Allow same parent to be selected multiple times
        p=probs
    )
    
    # 4. Return the actual parent chromosomes and their indices
    parents = ga_instance.population[selected_indices, :].copy()
    
    return parents, selected_indices

def stage_level_crossover(parents, offspring_size, ga_instance):
    offspring = []
    idx = 0
    
    # We iterate until we have enough offspring to fill the required size
    while len(offspring) < offspring_size[0]:
        # Select pair of parents
        parent1 = parents[idx % parents.shape[0]].copy()
        parent2 = parents[(idx + 1) % parents.shape[0]].copy()
        
        # Don't forget to hardcord stages_info!
        stages_info = [3, 4] 
        p_c = 0.2 # Probability of crossover occurring at all
        q_c = 0.3 # Probability of swapping a specific stage
        
        if np.random.rand() < p_c:
            start_bit = 0
            for K_s in stages_info:
                bits_in_stage = (K_s * (K_s - 1)) // 2
                
                # Each pair of corresponding stages are exchanged with probability q_c
                if np.random.rand() < q_c:
                    # Swap the segments
                    temp = parent1[start_bit : start_bit + bits_in_stage].copy()
                    parent1[start_bit : start_bit + bits_in_stage] = parent2[start_bit : start_bit + bits_in_stage]
                    parent2[start_bit : start_bit + bits_in_stage] = temp
                
                start_bit += bits_in_stage
        
        # 3. Add both created offspring to the list
        offspring.append(parent1)
        if len(offspring) < offspring_size[0]:
            offspring.append(parent2)
        
        idx += 2

    return np.array(offspring)


In [ ]:
# List to store history: [{'gen': 0, 'dna': [...], 'fitness': [...]}, ...]
ga_history = []

def on_generation(ga_instance):
    generation = ga_instance.generations_completed
    
    # Calculate ratio of unique binary strings in each generation
    population = ga_instance.population
    unique_strands = np.unique(population, axis=0)
    diversity_ratio = len(unique_strands) / len(population)
    
    # Get all chromosomes (binary strings) and their fitness scores
    population_dna = ga_instance.population.copy()
    fitness_scores = ga_instance.last_generation_fitness.copy()
    
    # Store the data
    ga_history.append({
        'generation': generation,
        'chromosomes': population_dna,
        'fitness': fitness_scores,
        'best_fitness': np.max(fitness_scores),
        'diversity_ratio': diversity_ratio
    })
    
    print(f"Generation {generation} saved. Best Fitness: {np.max(fitness_scores)}")

### Run the GA

In [ ]:
K_list = [3, 4]
# Calculate total bits: (3*2)/2 + (4*3)/2 = 3 + 6 = 9 bits
total_bits = sum([(k * (k - 1)) // 2 for k in K_list])

ga_instance = pygad.GA(
    on_generation = on_generation,
    num_generations=20,
    num_parents_mating=20,
    fitness_func=fitness_func,
    parallel_processing=["process",20], # Launch 20 parallel processes (1 / individual)
    sol_per_pop=20, #Individuals per generation
    num_genes=total_bits, 
    gene_space=[0, 1],  
    parent_selection_type=roulette_wheel_selection,
    crossover_type=stage_level_crossover,
    mutation_type="random",
    mutation_percent_genes=10, # Probability of a bit flipping
    save_best_solutions=True,
    save_solutions = True
)

# Start the evolutionary process
ga_instance.run()

# Retrieve the best binary string found
solution, solution_fitness, solution_idx = ga_instance.best_solution()
print(f"Best Architecture String: {solution}")

In [ ]:
# To extend by another X generations:
ga_instance.num_generations = 20 # Set the number of *additional* generations
ga_instance.run()

In [ ]:
#print(f"Best Architecture Acc: {solution_fitness}")
#print(ga_instance.solutions())
ga_instance.plot_fitness()

### Post-processing and analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract all best chromosomes from history
best_dna_history = [entry['chromosomes'][np.argmax(entry['fitness'])] for entry in ga_history]

# Heat map showing best binary string vs. generation
plt.figure(figsize=(10, 6))
sns.heatmap(best_dna_history, cmap='YlGnBu')
plt.xlabel("Bit Index (Node Connections)", fontsize=14)
plt.ylabel("Generation", fontsize=14)
plt.title("Evolution of Best Architecture DNA", fontsize=16)
plt.show()

# Plot diveristy ratio of each generation

generation = [entry['generation'] for entry in ga_history]
diversity_ratio = [entry['diversity_ratio'] for entry in ga_history]

plt.figure(figsize=(10, 6))
plt.xlabel('Generation', fontsize = 14)
plt.ylabel('Diversity Ratio', fontsize = 14)
plt.title('Diversity Ratio vs. Generation', fontsize = 16)
plt.plot(generation, diversity_ratio, marker='o',linewidth=3)
#plt.xticks(generation, fontsize=12)
plt.yticks(fontsize=12)

plt.grid(True)

In [ ]:
# Plot evolution of best bit string

# Stack the best chromosome from each generation
best_chromosomes = np.array([entry['chromosomes'][np.argmax(entry['fitness'])] for entry in ga_history])

plt.figure(figsize=(15, 10))
sns.heatmap(best_chromosomes, cmap='Greys', cbar=False)
plt.xlabel('Gene Index (Connection)')
plt.ylabel('Generation')
plt.title('Evolution of the Best Solution DNA')
plt.show()

In [ ]:
# Plot fitness dist. vs. generation
plot_data = []
for entry in ga_history:
    for f in entry['fitness']:
        plot_data.append({'Generation': entry['generation'], 'Fitness': f})

df = pd.DataFrame(plot_data)

plt.figure(figsize=(12, 6))
sns.boxplot(x='Generation', y='Fitness', data=df, palette='viridis')
plt.title('Fitness Distribution vs. Generation')
plt.show()

In [ ]:

# 1. Convert the list of solutions into a NumPy array
all_evaluations = np.array(ga_instance.solutions)

# 2. Find unique rows (individuals)
# axis=0 treats each chromosome as a single unit
unique_individuals = np.unique(all_evaluations, axis=0)

total_unique_count = len(unique_individuals)
total_evaluations = len(all_evaluations)

print(f"Total individuals processed: {total_evaluations}")
print(f"Total unique individuals: {total_unique_count}")
print(f"Redundancy rate: {100 * (1 - total_unique_count/total_evaluations):.2f}%")